<a href="https://colab.research.google.com/github/vallepua/1.HOUSE-PRICE-PREDICTION-USING-LINEAR-ML-MODEL/blob/main/DM_ll_PPROJECT_EPA_Air.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## DATA MINING-ll PROJECT (Advisor: Dr.BIN LUO)

### TEAM MEMBERS:

ANIL VALLEPU

VINEETHA BURUGUPALLI

PARAM VENKAT VIVEK KESIREDDY

### GPU Setup

In [1]:
!nvidia-smi
!free -h
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

Wed Apr 15 02:33:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

### Colab setup with A100 High RAM

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/STAT8240_EPA_Project'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/embeddings', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/logs', exist_ok=True)
print(f'Project dir: {PROJECT_DIR}')
!ls {PROJECT_DIR}

MessageError: Error: credential propagation was unsuccessful

In [4]:
%cd /content
!git clone https://github.com/blacksnail789521/Time-IMM.git
!git clone https://github.com/blacksnail789521/IMM-TSF.git

/content
Cloning into 'Time-IMM'...
remote: Enumerating objects: 230, done.
remote: Counting objects: 100% (230/230), done.
remote: Compressing objects: 100% (192/192), done.
remote: Total 230 (delta 30), reused 229 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (230/230), 15.12 MiB | 9.39 MiB/s, done.
Resolving deltas: 100% (30/30), done.
Cloning into 'IMM-TSF'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 111 (delta 22), reused 106 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (111/111), 171.57 KiB | 19.06 MiB/s, done.
Resolving deltas: 100% (22/22), done.


In [5]:
print('=== IMM-TSF top-level ===')
!ls /content/IMM-TSF
print()
print('=== Time-IMM top-level ===')
!ls /content/Time-IMM

=== IMM-TSF top-level ===
compute_text_embeddings.py  layers   main_all.py  models	    utils
fusions			    lib      main_all.sh  README.md
go.sh			    LICENSE  main.py	  requirements.txt

=== Time-IMM top-level ===
data  README.md  visualize_data.ipynb


In [6]:
!find /content/IMM-TSF -maxdepth 3 -type f \( -name 'README*' -o -name 'requirements*.txt' -o -name 'setup.py' -o -name 'pyproject.toml' -o -name 'main.py' -o -name 'run*.py' -o -name 'train*.py' -o -name '*.sh' -o -name '*.yaml' -o -name '*.yml' \) | head -50

/content/IMM-TSF/main.py
/content/IMM-TSF/README.md
/content/IMM-TSF/go.sh
/content/IMM-TSF/requirements.txt
/content/IMM-TSF/main_all.sh


### Accessing GIT README file

In [7]:
!cat /content/IMM-TSF/README.md

# IMM-TSF Benchmark Library

[![arXiv](https://img.shields.io/badge/arXiv-2506.10412-b31b1b.svg)](https://arxiv.org/abs/2506.10412)
[![GitHub Stars](https://img.shields.io/github/stars/blacksnail789521/IMM-TSF?style=social)](https://github.com/blacksnail789521/IMM-TSF/stargazers)
[![](https://img.shields.io/badge/Project-Website-blue?style=flat)](https://blacksnail789521.github.io/time-imm-project-page/)
[![How to Cite](https://img.shields.io/badge/Cite-bibtex-orange)](#citation)

<p align="center"><sub>
✨ If you find our <em>paper</em> useful, a <strong>star ⭐ on GitHub</strong> helps others discover it and keeps you updated on future releases.
</sub></p>

## Overview

Welcome to the **IMM-TSF** benchmark library, part of the Time-IMM dataset collection for NeurIPS 2025 Datasets & Benchmarks Track. This repository provides tools for loading irregular, multimodal time-series data and running reproducible forecasting benchmarks.

## Repository Structure

```
IMM-TSF/                    

In [8]:
%cd /content/IMM-TSF
!cat requirements.txt

/content/IMM-TSF
torch==2.7.0
pandas==2.2.3
scikit-learn==1.6.1
reformer_pytorch==1.4.4
stribor==0.1.0
matplotlib==3.10.3
seaborn==0.13.2
geotorch==0.3.0
transformers==4.51.3
prettytable==3.16.0
accelerate==1.6.0


In [9]:
!pip install -q torch==2.7.0 torchvision==0.22.0 torchaudio==2.7.0 --index-url https://download.pytorch.org/whl/cu126 2>&1 | tail -15

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 75.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 84.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 111.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 118.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 120.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 MB 79.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 87.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.3/201.3 MB 72.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 108.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 139.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.8/866.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [10]:
%cd /content/IMM-TSF
!python main.py --help 2>&1 | head -120

/content/IMM-TSF
Traceback (most recent call last):
  File "/content/IMM-TSF/main.py", line 23, in <module>
    from models.tPatchGNN import tPatchGNN
  File "/content/IMM-TSF/models/tPatchGNN.py", line 7, in <module>
    from layers.SelfAttention_Family import FullAttention, AttentionLayer
  File "/content/IMM-TSF/layers/SelfAttention_Family.py", line 6, in <module>
    from reformer_pytorch import LSHSelfAttention
ModuleNotFoundError: No module named 'reformer_pytorch'


## Loading the EPA-Air Dataset from GIT repo

In [11]:
from google.colab import files
print("Upload your kaggle.json file:")
uploaded = files.upload()

Upload your kaggle.json file:


Saving time_series.csv to time_series.csv


In [12]:
!pip install -q kaggle
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!ls -la ~/.kaggle/

mv: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
total 12
drwxr-xr-x 2 root root 4096 Apr 15 02:37 .
drwx------ 1 root root 4096 Apr 15 02:37 ..


In [13]:
!mkdir -p /content/IMM-TSF/data
%cd /content/IMM-TSF/data
!kaggle datasets download -d blacksnail789521/time-imm --unzip
!ls

/content/IMM-TSF/data
Dataset URL: https://www.kaggle.com/datasets/blacksnail789521/time-imm
License(s): Attribution 4.0 International (CC BY 4.0)
100% 15.4M/15.4M [00:03<00:00, 5.07MB/s]

CESNET	      EPA-Air  GDELT   MIMIC	  RepoHealth
ClusterTrace  FNSPID   ILINet  README.md  StudentLife


## Note: **** Loaded all cities EPA-Air dataset which authors used in the paper ****

In [14]:
!ls /content/IMM-TSF/data/EPA-Air/processed/ 2>/dev/null || ls /content/IMM-TSF/data/

Bexar	Denver	      Los_Angeles  Philadelphia
Dallas	Hillsborough  Maricopa	   Richmond


In [15]:
import pandas as pd
from pathlib import Path

epa_root = Path('/content/IMM-TSF/data/EPA-Air/processed')
counties = sorted([p.name for p in epa_root.iterdir() if p.is_dir()])
print(f'Number of counties: {len(counties)}')
print(f'Counties: {counties}')

total_obs = 0
total_text = 0
for c in counties:
    ts = pd.read_csv(epa_root / c / 'time_series.csv')
    txt = pd.read_csv(epa_root / c / 'text.csv')
    total_obs += len(ts)
    total_text += len(txt)
    print(f'  {c}: ts={ts.shape}, text={txt.shape}, ts_cols={list(ts.columns)}')

print(f'\nTotal observations: {total_obs}')
print(f'Total text entries: {total_text}')
print(f'\nPaper Table 1 expects: 49,552 obs, 1,244 text entries, 4 features')

Number of counties: 8
Counties: ['Bexar', 'Dallas', 'Denver', 'Hillsborough', 'Los_Angeles', 'Maricopa', 'Philadelphia', 'Richmond']
  Bexar: ts=(4369, 6), text=(17, 3), ts_cols=['date_time', 'record_id', 'temp', 'pm2_5', 'aqi', 'ozone']
  Dallas: ts=(4369, 6), text=(209, 3), ts_cols=['date_time', 'record_id', 'temp', 'pm2_5', 'aqi', 'ozone']
  Denver: ts=(5957, 6), text=(213, 3), ts_cols=['date_time', 'record_id', 'temp', 'pm2_5', 'aqi', 'ozone']
  Hillsborough: ts=(4369, 6), text=(73, 3), ts_cols=['date_time', 'record_id', 'temp', 'pm2_5', 'aqi', 'ozone']
  Los_Angeles: ts=(5851, 6), text=(222, 3), ts_cols=['date_time', 'record_id', 'temp', 'pm2_5', 'aqi', 'ozone']
  Maricopa: ts=(6567, 6), text=(99, 3), ts_cols=['date_time', 'record_id', 'temp', 'pm2_5', 'aqi', 'ozone']
  Philadelphia: ts=(4337, 6), text=(203, 3), ts_cols=['date_time', 'record_id', 'temp', 'pm2_5', 'aqi', 'ozone']
  Richmond: ts=(6487, 6), text=(208, 3), ts_cols=['date_time', 'record_id', 'temp', 'pm2_5', 'aqi', 'oz

In [16]:
import pandas as pd
from pathlib import Path

epa_root = Path('/content/IMM-TSF/data/EPA-Air/processed')
counties = sorted([p.name for p in epa_root.iterdir() if p.is_dir()])

feature_cols = ['temp', 'pm2_5', 'aqi', 'ozone']
total_rows = 0
total_nonnull_cells = 0

for c in counties:
    ts = pd.read_csv(epa_root / c / 'time_series.csv')
    rows = len(ts)
    nonnull = ts[feature_cols].notna().sum().sum()
    total_rows += rows
    total_nonnull_cells += nonnull
    null_pct = ts[feature_cols].isna().mean().mean() * 100
    print(f'  {c}: rows={rows}, non-null cells={nonnull}, missing={null_pct:.1f}%')

print(f'\nTotal rows across all counties: {total_rows}')
print(f'Total non-null feature cells:    {total_nonnull_cells}')
print(f'Paper Table 1 expects:           49,552 observations')
print(f'\nIf "non-null cells" matches 49,552 -> paper counts cells, not rows (benign)')
print(f'If neither matches -> dataset version differs from paper (note in report)')

  Bexar: rows=4369, non-null cells=5123, missing=70.7%
  Dallas: rows=4369, non-null cells=5123, missing=70.7%
  Denver: rows=5957, non-null cells=6998, missing=70.6%
  Hillsborough: rows=4369, non-null cells=5123, missing=70.7%
  Los_Angeles: rows=5851, non-null cells=6860, missing=70.7%
  Maricopa: rows=6567, non-null cells=7702, missing=70.7%
  Philadelphia: rows=4337, non-null cells=5083, missing=70.7%
  Richmond: rows=6487, non-null cells=7540, missing=70.9%

Total rows across all counties: 42306
Total non-null feature cells:    49552
Paper Table 1 expects:           49,552 observations

If "non-null cells" matches 49,552 -> paper counts cells, not rows (benign)
If neither matches -> dataset version differs from paper (note in report)


In [17]:
sample = pd.read_csv(epa_root / 'Los_Angeles' / 'time_series.csv')
print('Shape:', sample.shape)
print('\nFirst 10 rows:')
print(sample.head(10))
print('\nMissing per column:')
print(sample.isna().sum())
print('\ndate_time range:', sample['date_time'].min(), '->', sample['date_time'].max())

Shape: (5851, 6)

First 10 rows:
             date_time    record_id  temp    pm2_5    aqi     ozone
0  2024-01-01 00:00:00  Los_Angeles  36.2  49.9550  104.0  0.021742
1  2024-01-01 01:00:00  Los_Angeles  35.4      NaN    NaN       NaN
2  2024-01-01 02:00:00  Los_Angeles  34.7      NaN    NaN       NaN
3  2024-01-01 03:00:00  Los_Angeles  34.1      NaN    NaN       NaN
4  2024-01-01 04:00:00  Los_Angeles  32.7      NaN    NaN       NaN
5  2024-01-01 05:00:00  Los_Angeles  32.8      NaN    NaN       NaN
6  2024-01-01 06:00:00  Los_Angeles  31.4      NaN    NaN       NaN
7  2024-01-01 07:00:00  Los_Angeles  30.9      NaN    NaN       NaN
8  2024-01-01 08:00:00  Los_Angeles  34.9   7.4425    NaN       NaN
9  2024-01-01 09:00:00  Los_Angeles  39.3      NaN    NaN       NaN

Missing per column:
date_time       0
record_id       3
temp            3
pm2_5        5119
aqi          5606
ozone        5816
dtype: int64

date_time range: 2024-01-01 00:00:00 -> 2024-09-01 00:00:00


In [18]:
text_sample = pd.read_csv(epa_root / 'Los_Angeles' / 'text.csv')
print('Text shape:', text_sample.shape)
print('Columns:', list(text_sample.columns))
print('\nFirst 3 entries:')
for i in range(min(3, len(text_sample))):
    print(f'\n--- Entry {i} ---')
    for col in text_sample.columns:
        val = str(text_sample.iloc[i][col])
        if len(val) > 200:
            val = val[:200] + '...'
        print(f'  {col}: {val}')

Text shape: (222, 3)
Columns: ['date_time', 'record_id', 'text']

First 3 entries:

--- Entry 0 ---
  date_time: 2024-01-01 00:35:32
  record_id: Los_Angeles
  text: Southern California experienced high surf with waves reaching up to 20 feet, prompting a temporary evacuation warning along parts of the coast, particularly near the Pacific Coast Highway. The surf, d...

--- Entry 1 ---
  date_time: 2024-01-01 00:35:54
  record_id: Los_Angeles
  text: Southern California's Ventura County issued a temporary evacuation warning due to high surf reaching up to 20 feet, causing significant coastal impacts. The waves, described as an "extraordinary event...

--- Entry 2 ---
  date_time: 2024-01-01 00:36:42
  record_id: Los_Angeles
  text: Southern California's Ventura County issued a temporary evacuation warning due to high surf reaching up to 20 feet (6 meters), causing coastal hazards and street closures. The warning was later lifted...


In [19]:
!pip install -q reformer_pytorch==1.4.4 stribor==0.1.0 geotorch==0.3.0 2>&1 | tail -20

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 6.9 MB/s eta 0:00:00


In [20]:
%cd /content/IMM-TSF
!python main.py \
  --dataset EPA-Air \
  --data_root /content/IMM-TSF/data \
  --model DLinear \
  --history 7 \
  --pred_window 7 \
  --stride 7 \
  --time_unit days \
  --split_method sample \
  --seed 42 \
  --gpu 0 \
  --batch_size 8 \
  --lr 1e-3 \
  --epoch 50 \
  --patience 10 \
  2>&1 | tail -60

/content/IMM-TSF
Train - Loss (one batch): 0.59749
Val - Loss, MSE, MAE: 0.75206, 0.75206, 0.57694
Test - Best epoch, Loss, MSE, MAE: 0, 0.54379, 0.54379, 0.53738
Time spent: 0.20s
100%|██████████| 6/6 [00:00<00:00, 193.55it/s]
- Epoch 004, ExpID 95356
Train - Loss (one batch): 0.53136
Val - Loss, MSE, MAE: 0.74953, 0.74953, 0.57302
Test - Best epoch, Loss, MSE, MAE: 0, 0.54379, 0.54379, 0.53738
Time spent: 0.20s
100%|██████████| 6/6 [00:00<00:00, 193.16it/s]
- Epoch 005, ExpID 95356
Train - Loss (one batch): 0.44465
Val - Loss, MSE, MAE: 0.77039, 0.77039, 0.58117
Test - Best epoch, Loss, MSE, MAE: 0, 0.54379, 0.54379, 0.53738
Time spent: 0.20s
100%|██████████| 6/6 [00:00<00:00, 192.00it/s]
- Epoch 006, ExpID 95356
Train - Loss (one batch): 0.47725
Val - Loss, MSE, MAE: 0.74140, 0.74140, 0.57198
Test - Best epoch, Loss, MSE, MAE: 0, 0.54379, 0.54379, 0.53738
Time spent: 0.20s
100%|██████████| 6/6 [00:00<00:00, 193.76it/s]
- Epoch 007, ExpID 95356
Train - Loss (one batch): 0.47475
Val -

In [21]:
!ls /content/IMM-TSF/data/EPA-Air/processed/Los_Angeles/

text.csv  time_series.csv


In [22]:
!sed -n '110,160p' /content/IMM-TSF/compute_text_embeddings.py

    # * Parameters
    data_name_list = [
        "GDELT",  # type 1.1
        "RepoHealth",  # type 1.2
        "MIMIC",  # type 1.3
        "FNSPID",  # type 2.1
        # "ClusterTrace",  # type 2.2
        "StudentLife",  # type 2.3
        "ILINet",  # type 3.1
        "CESNET",  # type 3.2
        "EPA-Air",  # type 3.3
    ]

    llm_model_fusion = "GPT2"
    # llm_model_fusion = "GPT2XL"
    # llm_model_fusion = "BERT"
    # llm_model_fusion = "Llama"
    # llm_model_fusion = "DeepSeek"
    # llm_layers_fusion = 6
    llm_layers_fusion = None
    max_length = 512 if llm_model_fusion == "BERT" else 1024
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print(f"### LLM model: {llm_model_fusion} ###")

    # * Update max_length if needed
    # context_window_size = get_context_window_size(llm_model_fusion, device)
    # if max_length > context_window_size:
    #     print(
    #         f"Overriding max_length from {max_length} to {context_window_size}"
    #       

In [23]:
# Back up original first (good habit before editing library files)
!cp /content/IMM-TSF/compute_text_embeddings.py /content/IMM-TSF/compute_text_embeddings.py.bak

# Use Python to rewrite the list in-place — safer than sed for multi-line edits
import re
path = '/content/IMM-TSF/compute_text_embeddings.py'
with open(path, 'r') as f:
    src = f.read()

# Replace the entire data_name_list block with a single-entry list
new_list = '''data_name_list = [
        "EPA-Air",  # type 3.3 (only target for this run)
    ]'''

pattern = r'data_name_list = \[.*?\]'
patched = re.sub(pattern, new_list, src, count=1, flags=re.DOTALL)

with open(path, 'w') as f:
    f.write(patched)

# Verify the edit took
!grep -A3 "data_name_list = " /content/IMM-TSF/compute_text_embeddings.py | head -10

    data_name_list = [
        "EPA-Air",  # type 3.3 (only target for this run)
    ]



In [24]:
%cd /content/IMM-TSF
!python compute_text_embeddings.py 2>&1 | tail -30

/content/IMM-TSF
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loaded GPT2 with full layers for fusion.
[1/8] Processing record: Bexar
Embedding notes for record 'Bexar'...
Wrote embeddings to data/EPA-Air/processed/Bexar/text_embeddings_model=GPT2_layers=full_maxlen=1024.pt
[2/8] Processing record: Dallas
Embedding notes for record 'Dallas'...
Wrote embeddings to data/EPA-Air/processed/Dallas/text_embeddings_model=GPT2_layers=full_maxlen=1024.pt
[3/8] Processing record: Denver
Embedding notes for record 'Denver'...
Wrote embeddings to data/EPA-Air/processed/Denver/text_embeddings_model=GPT2_layers=full_maxlen=1024.pt
[4/8] Processing record: Hillsborough
Embedding notes for record 'Hillsborough'...
Wrote embeddings to data/EPA-Air/processed/Hillsborough/text_embeddings_model=GPT2_layers=full_maxlen=1024.pt
[5/8] Processing record: L

In [25]:
!find /content/IMM-TSF/data/EPA-Air/processed -name "*.pt" | sort

/content/IMM-TSF/data/EPA-Air/processed/Bexar/text_embeddings_model=GPT2_layers=full_maxlen=1024.pt
/content/IMM-TSF/data/EPA-Air/processed/Dallas/text_embeddings_model=GPT2_layers=full_maxlen=1024.pt
/content/IMM-TSF/data/EPA-Air/processed/Denver/text_embeddings_model=GPT2_layers=full_maxlen=1024.pt
/content/IMM-TSF/data/EPA-Air/processed/Hillsborough/text_embeddings_model=GPT2_layers=full_maxlen=1024.pt
/content/IMM-TSF/data/EPA-Air/processed/Los_Angeles/text_embeddings_model=GPT2_layers=full_maxlen=1024.pt
/content/IMM-TSF/data/EPA-Air/processed/Maricopa/text_embeddings_model=GPT2_layers=full_maxlen=1024.pt
/content/IMM-TSF/data/EPA-Air/processed/Philadelphia/text_embeddings_model=GPT2_layers=full_maxlen=1024.pt
/content/IMM-TSF/data/EPA-Air/processed/Richmond/text_embeddings_model=GPT2_layers=full_maxlen=1024.pt


In [26]:
# Check the current state of the script
!sed -n '120,135p' /content/IMM-TSF/compute_text_embeddings.py

    # llm_layers_fusion = 6
    llm_layers_fusion = None
    max_length = 512 if llm_model_fusion == "BERT" else 1024
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print(f"### LLM model: {llm_model_fusion} ###")

    # * Update max_length if needed
    # context_window_size = get_context_window_size(llm_model_fusion, device)
    # if max_length > context_window_size:
    #     print(
    #         f"Overriding max_length from {max_length} to {context_window_size}"
    #         " to match the LLM model's context window size."
    #     )
    #     max_length = context_window_size



In [27]:
import re
path = '/content/IMM-TSF/compute_text_embeddings.py'
with open(path, 'r') as f:
    src = f.read()

# Swap: comment out 'llm_layers_fusion = None', uncomment 'llm_layers_fusion = 6'
src = src.replace('    # llm_layers_fusion = 6', '    llm_layers_fusion = 6')
src = src.replace('    llm_layers_fusion = None', '    # llm_layers_fusion = None')

with open(path, 'w') as f:
    f.write(src)

# Verify
!grep "llm_layers_fusion" /content/IMM-TSF/compute_text_embeddings.py | head -5

    llm_layers_fusion: int | None,
    and save text_embeddings_{llm_model_fusion}_{llm_layers_fusion or 'full'}.pt
      llm_layers_fusion: number of layers to keep, or None for all
        llm_layers_fusion,
            f"_layers={llm_layers_fusion or 'full'}"


In [28]:
%cd /content/IMM-TSF
!python compute_text_embeddings.py 2>&1 | tail -20

/content/IMM-TSF
Embedding notes for record 'Dallas'...
Wrote embeddings to data/EPA-Air/processed/Dallas/text_embeddings_model=GPT2_layers=6_maxlen=1024.pt
[3/8] Processing record: Denver
Embedding notes for record 'Denver'...
Wrote embeddings to data/EPA-Air/processed/Denver/text_embeddings_model=GPT2_layers=6_maxlen=1024.pt
[4/8] Processing record: Hillsborough
Embedding notes for record 'Hillsborough'...
Wrote embeddings to data/EPA-Air/processed/Hillsborough/text_embeddings_model=GPT2_layers=6_maxlen=1024.pt
[5/8] Processing record: Los_Angeles
Embedding notes for record 'Los_Angeles'...
Wrote embeddings to data/EPA-Air/processed/Los_Angeles/text_embeddings_model=GPT2_layers=6_maxlen=1024.pt
[6/8] Processing record: Maricopa
Embedding notes for record 'Maricopa'...
Wrote embeddings to data/EPA-Air/processed/Maricopa/text_embeddings_model=GPT2_layers=6_maxlen=1024.pt
[7/8] Processing record: Philadelphia
Embedding notes for record 'Philadelphia'...
Wrote embeddings to data/EPA-Air/

In [29]:
!ls /content/IMM-TSF/data/EPA-Air/processed/Los_Angeles/*.pt

'/content/IMM-TSF/data/EPA-Air/processed/Los_Angeles/text_embeddings_model=GPT2_layers=6_maxlen=1024.pt'
'/content/IMM-TSF/data/EPA-Air/processed/Los_Angeles/text_embeddings_model=GPT2_layers=full_maxlen=1024.pt'


## 1. DLiner





In [30]:
%cd /content/IMM-TSF
!python main.py \
  --dataset EPA-Air \
  --data_root /content/IMM-TSF/data \
  --model DLinear \
  --history 7 \
  --pred_window 7 \
  --stride 7 \
  --time_unit days \
  --split_method sample \
  --enable_text \
  --use_text_embeddings \
  --TTF_module TTF_RecAvg \
  --MMF_module MMF_GR_Add \
  --llm_model_fusion GPT2 \
  --seed 42 \
  --gpu 0 \
  --batch_size 8 \
  --lr 1e-3 \
  --epoch 50 \
  --patience 10 \
  2>&1 | tail -60

/content/IMM-TSF
Train - Loss (one batch): 0.56549
Val - Loss, MSE, MAE: 0.74316, 0.74316, 0.58157
Test - Best epoch, Loss, MSE, MAE: 7, 0.50896, 0.50896, 0.51893
Time spent: 0.45s
100%|██████████| 5/5 [00:00<00:00, 135.16it/s]
- Epoch 011, ExpID 68981
Train - Loss (one batch): 0.27462
Val - Loss, MSE, MAE: 0.77421, 0.77421, 0.59508
Test - Best epoch, Loss, MSE, MAE: 7, 0.50896, 0.50896, 0.51893
Time spent: 0.44s
100%|██████████| 5/5 [00:00<00:00, 134.15it/s]
- Epoch 012, ExpID 68981
Train - Loss (one batch): 0.31879
Val - Loss, MSE, MAE: 0.76695, 0.76695, 0.58780
Test - Best epoch, Loss, MSE, MAE: 7, 0.50896, 0.50896, 0.51893
Time spent: 0.43s
100%|██████████| 5/5 [00:00<00:00, 131.45it/s]
- Epoch 013, ExpID 68981
Train - Loss (one batch): 0.23835
Val - Loss, MSE, MAE: 0.76073, 0.76073, 0.59092
Test - Best epoch, Loss, MSE, MAE: 7, 0.50896, 0.50896, 0.51893
Time spent: 0.43s
100%|██████████| 5/5 [00:00<00:00, 132.53it/s]
- Epoch 014, ExpID 68981
Train - Loss (one batch): 0.46143
Val -

In [31]:
import json
from datetime import datetime
from pathlib import Path

RESULTS_FILE = '/content/drive/MyDrive/STAT8240_EPA_Project/results/runs.jsonl'
Path(RESULTS_FILE).parent.mkdir(parents=True, exist_ok=True)

def log_run(model, modal, ttf, mmf, text_encoder, mse, mae, best_epoch, notes=''):
    entry = {
        'timestamp': datetime.now().isoformat(),
        'model': model,
        'modal': modal,
        'ttf': ttf,
        'mmf': mmf,
        'text_encoder': text_encoder,
        'mse': mse,
        'mae': mae,
        'best_epoch': best_epoch,
        'seed': 42,
        'dataset': 'EPA-Air',
        'notes': notes,
    }
    with open(RESULTS_FILE, 'a') as f:
        f.write(json.dumps(entry) + '\n')
    print(f'Logged: {model} ({modal}) MSE={mse:.4f} MAE={mae:.4f}')

# Log the two runs we just completed
log_run('DLinear', 'uni', None, None, None, 0.5438, 0.5374, 0, 'Phase 1 smoke test')
log_run('DLinear', 'multi', 'RecAvg', 'GR_Add', 'GPT2', 0.5090, 0.5189, 7, 'Phase 1 reproduction')

!cat {RESULTS_FILE}

Logged: DLinear (uni) MSE=0.5438 MAE=0.5374
Logged: DLinear (multi) MSE=0.5090 MAE=0.5189
{"timestamp": "2026-04-15T02:41:21.117014", "model": "DLinear", "modal": "uni", "ttf": null, "mmf": null, "text_encoder": null, "mse": 0.5438, "mae": 0.5374, "best_epoch": 0, "seed": 42, "dataset": "EPA-Air", "notes": "Phase 1 smoke test"}
{"timestamp": "2026-04-15T02:41:21.117337", "model": "DLinear", "modal": "multi", "ttf": "RecAvg", "mmf": "GR_Add", "text_encoder": "GPT2", "mse": 0.509, "mae": 0.5189, "best_epoch": 7, "seed": 42, "dataset": "EPA-Air", "notes": "Phase 1 reproduction"}


## 2. Informer

In [32]:
%cd /content/IMM-TSF
!python main.py \
  --dataset EPA-Air \
  --data_root /content/IMM-TSF/data \
  --model Informer \
  --history 7 \
  --pred_window 7 \
  --stride 7 \
  --time_unit days \
  --split_method sample \
  --seed 42 \
  --gpu 0 \
  --batch_size 8 \
  --lr 1e-3 \
  --epoch 50 \
  --patience 10 \
  2>&1 | tail -40

/content/IMM-TSF
- Epoch 035, ExpID 19007
Train - Loss (one batch): 0.42897
Val - Loss, MSE, MAE: 0.84117, 0.84117, 0.62594
Test - Best epoch, Loss, MSE, MAE: 29, 0.64717, 0.64717, 0.60302
Time spent: 1.79s
100%|██████████| 6/6 [00:00<00:00, 57.94it/s]
- Epoch 036, ExpID 19007
Train - Loss (one batch): 0.41451
Val - Loss, MSE, MAE: 0.83447, 0.83447, 0.63012
Test - Best epoch, Loss, MSE, MAE: 29, 0.64717, 0.64717, 0.60302
Time spent: 1.77s
100%|██████████| 6/6 [00:00<00:00, 58.08it/s]
- Epoch 037, ExpID 19007
Train - Loss (one batch): 0.62926
Val - Loss, MSE, MAE: 0.81015, 0.81015, 0.62511
Test - Best epoch, Loss, MSE, MAE: 29, 0.64717, 0.64717, 0.60302
Time spent: 1.78s
100%|██████████| 6/6 [00:00<00:00, 58.08it/s]
- Epoch 038, ExpID 19007
Train - Loss (one batch): 0.72906
Val - Loss, MSE, MAE: 0.81138, 0.81138, 0.62568
Test - Best epoch, Loss, MSE, MAE: 29, 0.64717, 0.64717, 0.60302
Time spent: 1.75s
100%|██████████| 6/6 [00:00<00:00, 57.90it/s]
- Epoch 039, ExpID 19007
Train - Loss (

In [33]:
%cd /content/IMM-TSF
!python main.py \
  --dataset EPA-Air \
  --data_root /content/IMM-TSF/data \
  --model Informer \
  --history 7 \
  --pred_window 7 \
  --stride 7 \
  --time_unit days \
  --split_method sample \
  --enable_text \
  --use_text_embeddings \
  --TTF_module TTF_RecAvg \
  --MMF_module MMF_GR_Add \
  --llm_model_fusion GPT2 \
  --seed 42 \
  --gpu 0 \
  --batch_size 8 \
  --lr 1e-3 \
  --epoch 50 \
  --patience 10 \
  2>&1 | tail -40

/content/IMM-TSF
- Epoch 032, ExpID 95226
Train - Loss (one batch): 0.42365
Val - Loss, MSE, MAE: 0.80920, 0.80920, 0.63758
Test - Best epoch, Loss, MSE, MAE: 26, 0.59369, 0.59369, 0.57501
Time spent: 2.17s
100%|██████████| 5/5 [00:00<00:00, 48.78it/s]
- Epoch 033, ExpID 95226
Train - Loss (one batch): 0.43498
Val - Loss, MSE, MAE: 0.80875, 0.80875, 0.63438
Test - Best epoch, Loss, MSE, MAE: 26, 0.59369, 0.59369, 0.57501
Time spent: 2.12s
100%|██████████| 5/5 [00:00<00:00, 48.33it/s]
- Epoch 034, ExpID 95226
Train - Loss (one batch): 0.22961
Val - Loss, MSE, MAE: 0.80579, 0.80579, 0.64234
Test - Best epoch, Loss, MSE, MAE: 26, 0.59369, 0.59369, 0.57501
Time spent: 2.14s
100%|██████████| 5/5 [00:00<00:00, 48.84it/s]
- Epoch 035, ExpID 95226
Train - Loss (one batch): 0.23686
Val - Loss, MSE, MAE: 0.83491, 0.83491, 0.64261
Test - Best epoch, Loss, MSE, MAE: 26, 0.59369, 0.59369, 0.57501
Time spent: 2.10s
100%|██████████| 5/5 [00:00<00:00, 48.50it/s]
- Epoch 036, ExpID 95226
Train - Loss (

In [34]:
log_run('Informer', 'uni',   None,     None,     None,   0.6472, 0.6030, 29, 'reproduction')
log_run('Informer', 'multi', 'RecAvg', 'GR_Add', 'GPT2', 0.5937, 0.5750, 26, 'reproduction')

Logged: Informer (uni) MSE=0.6472 MAE=0.6030
Logged: Informer (multi) MSE=0.5937 MAE=0.5750


# 3. PatchTST

In [35]:
%cd /content/IMM-TSF
!python main.py \
  --dataset EPA-Air \
  --data_root /content/IMM-TSF/data \
  --model PatchTST \
  --history 7 \
  --pred_window 7 \
  --stride 7 \
  --time_unit days \
  --split_method sample \
  --seed 42 \
  --gpu 0 \
  --batch_size 8 \
  --lr 1e-3 \
  --epoch 50 \
  --patience 10 \
  2>&1 | tail -30

/content/IMM-TSF
Test - Best epoch, Loss, MSE, MAE: 10, 0.62972, 0.62972, 0.59914
Time spent: 0.88s
100%|██████████| 6/6 [00:00<00:00, 90.22it/s]
- Epoch 018, ExpID 32909
Train - Loss (one batch): 0.55053
Val - Loss, MSE, MAE: 1.07453, 1.07453, 0.71695
Test - Best epoch, Loss, MSE, MAE: 10, 0.62972, 0.62972, 0.59914
Time spent: 0.88s
100%|██████████| 6/6 [00:00<00:00, 89.41it/s]
- Epoch 019, ExpID 32909
Train - Loss (one batch): 0.74067
Val - Loss, MSE, MAE: 1.11894, 1.11894, 0.74369
Test - Best epoch, Loss, MSE, MAE: 10, 0.62972, 0.62972, 0.59914
Time spent: 0.88s
100%|██████████| 6/6 [00:00<00:00, 89.70it/s]
- Epoch 020, ExpID 32909
Train - Loss (one batch): 0.68779
Val - Loss, MSE, MAE: 0.95558, 0.95558, 0.70450
Test - Best epoch, Loss, MSE, MAE: 10, 0.62972, 0.62972, 0.59914
Time spent: 0.89s
Exp has been early stopped!
loss: 0.6297236084938049
mse: 0.6297236084938049
mae: 0.5991351008415222
rmse: 0.7935512661933899
mape: -1.1023776531219482
### Done ###


In [36]:
%cd /content/IMM-TSF
!python main.py \
  --dataset EPA-Air \
  --data_root /content/IMM-TSF/data \
  --model PatchTST \
  --history 7 \
  --pred_window 7 \
  --stride 7 \
  --time_unit days \
  --split_method sample \
  --enable_text \
  --use_text_embeddings \
  --TTF_module TTF_RecAvg \
  --MMF_module MMF_GR_Add \
  --llm_model_fusion GPT2 \
  --seed 42 \
  --gpu 0 \
  --batch_size 8 \
  --lr 1e-3 \
  --epoch 50 \
  --patience 10 \
  2>&1 | tail -30

/content/IMM-TSF
Test - Best epoch, Loss, MSE, MAE: 29, 0.67971, 0.67971, 0.60937
Time spent: 1.18s
100%|██████████| 5/5 [00:00<00:00, 70.11it/s]
- Epoch 037, ExpID 98472
Train - Loss (one batch): 0.31097
Val - Loss, MSE, MAE: 1.01571, 1.01571, 0.71016
Test - Best epoch, Loss, MSE, MAE: 29, 0.67971, 0.67971, 0.60937
Time spent: 1.19s
100%|██████████| 5/5 [00:00<00:00, 69.17it/s]
- Epoch 038, ExpID 98472
Train - Loss (one batch): 1.94913
Val - Loss, MSE, MAE: 0.93617, 0.93617, 0.69474
Test - Best epoch, Loss, MSE, MAE: 29, 0.67971, 0.67971, 0.60937
Time spent: 1.15s
100%|██████████| 5/5 [00:00<00:00, 71.89it/s]
- Epoch 039, ExpID 98472
Train - Loss (one batch): 2.06712
Val - Loss, MSE, MAE: 0.99443, 0.99443, 0.71020
Test - Best epoch, Loss, MSE, MAE: 29, 0.67971, 0.67971, 0.60937
Time spent: 1.15s
Exp has been early stopped!
loss: 0.6797087788581848
mse: 0.6797087788581848
mae: 0.6093711853027344
rmse: 0.8244445323944092
mape: -1.2826120853424072
### Done ###


In [37]:
log_run('PatchTST', 'uni',   None,     None,     None,   0.6297, 0.5991, 10, 'reproduction')
log_run('PatchTST', 'multi', 'RecAvg', 'GR_Add', 'GPT2', 0.6797, 0.6094, 29, 'reproduction - text hurts, matches paper direction')

Logged: PatchTST (uni) MSE=0.6297 MAE=0.5991
Logged: PatchTST (multi) MSE=0.6797 MAE=0.6094


# 4 TimesNet

In [38]:
%cd /content/IMM-TSF
!python main.py \
  --dataset EPA-Air \
  --data_root /content/IMM-TSF/data \
  --model TimesNet \
  --history 7 \
  --pred_window 7 \
  --stride 7 \
  --time_unit days \
  --split_method sample \
  --seed 42 \
  --gpu 0 \
  --batch_size 8 \
  --lr 1e-3 \
  --epoch 50 \
  --patience 10 \
  2>&1 | tail -30

/content/IMM-TSF
Test - Best epoch, Loss, MSE, MAE: 11, 0.58881, 0.58881, 0.56843
Time spent: 27.04s
100%|██████████| 6/6 [00:03<00:00,  1.62it/s]
- Epoch 019, ExpID 5975
Train - Loss (one batch): 0.58577
Val - Loss, MSE, MAE: 1.35300, 1.35300, 0.84206
Test - Best epoch, Loss, MSE, MAE: 11, 0.58881, 0.58881, 0.56843
Time spent: 27.11s
100%|██████████| 6/6 [00:03<00:00,  1.62it/s]
- Epoch 020, ExpID 5975
Train - Loss (one batch): 0.48931
Val - Loss, MSE, MAE: 1.10706, 1.10706, 0.73397
Test - Best epoch, Loss, MSE, MAE: 11, 0.58881, 0.58881, 0.56843
Time spent: 26.98s
100%|██████████| 6/6 [00:03<00:00,  1.88it/s]
- Epoch 021, ExpID 5975
Train - Loss (one batch): 0.53673
Val - Loss, MSE, MAE: 1.27529, 1.27529, 0.83545
Test - Best epoch, Loss, MSE, MAE: 11, 0.58881, 0.58881, 0.56843
Time spent: 25.85s
Exp has been early stopped!
loss: 0.5888111591339111
mse: 0.5888111591339111
mae: 0.5684305429458618
rmse: 0.7673403024673462
mape: -0.36858752369880676
### Done ###


In [39]:
%cd /content/IMM-TSF
!python main.py \
  --dataset EPA-Air \
  --data_root /content/IMM-TSF/data \
  --model TimesNet \
  --history 7 \
  --pred_window 7 \
  --stride 7 \
  --time_unit days \
  --split_method sample \
  --enable_text \
  --use_text_embeddings \
  --TTF_module TTF_RecAvg \
  --MMF_module MMF_GR_Add \
  --llm_model_fusion GPT2 \
  --seed 42 \
  --gpu 0 \
  --batch_size 8 \
  --lr 1e-3 \
  --epoch 50 \
  --patience 10 \
  2>&1 | tail -30

/content/IMM-TSF
Test - Best epoch, Loss, MSE, MAE: 7, 0.57552, 0.57552, 0.55227
Time spent: 28.30s
100%|██████████| 5/5 [00:03<00:00,  1.52it/s]
- Epoch 015, ExpID 3037
Train - Loss (one batch): 0.70902
Val - Loss, MSE, MAE: 1.90566, 1.90566, 1.03695
Test - Best epoch, Loss, MSE, MAE: 7, 0.57552, 0.57552, 0.55227
Time spent: 27.68s
100%|██████████| 5/5 [00:03<00:00,  1.52it/s]
- Epoch 016, ExpID 3037
Train - Loss (one batch): 0.75997
Val - Loss, MSE, MAE: 1.82418, 1.82418, 0.99380
Test - Best epoch, Loss, MSE, MAE: 7, 0.57552, 0.57552, 0.55227
Time spent: 27.15s
100%|██████████| 5/5 [00:03<00:00,  1.50it/s]
- Epoch 017, ExpID 3037
Train - Loss (one batch): 0.47182
Val - Loss, MSE, MAE: 1.35071, 1.35071, 0.85977
Test - Best epoch, Loss, MSE, MAE: 7, 0.57552, 0.57552, 0.55227
Time spent: 27.36s
Exp has been early stopped!
loss: 0.5755187273025513
mse: 0.5755187273025513
mae: 0.5522687435150146
rmse: 0.7586295008659363
mape: -0.8159303665161133
### Done ###


In [40]:
log_run('TimesNet', 'uni',   None,     None,     None,   0.5888, 0.5684, 11, 'reproduction - slightly higher than paper')
log_run('TimesNet', 'multi', 'RecAvg', 'GR_Add', 'GPT2', 0.5755, 0.5523,  7, 'reproduction - multi beats uni, reverses paper direction')

Logged: TimesNet (uni) MSE=0.5888 MAE=0.5684
Logged: TimesNet (multi) MSE=0.5755 MAE=0.5523


# 5. TimeMixer

In [41]:
%cd /content/IMM-TSF
!python main.py \
  --dataset EPA-Air \
  --data_root /content/IMM-TSF/data \
  --model TimeMixer \
  --history 7 \
  --pred_window 7 \
  --stride 7 \
  --time_unit days \
  --split_method sample \
  --seed 42 \
  --gpu 0 \
  --batch_size 8 \
  --lr 1e-3 \
  --epoch 50 \
  --patience 10 \
  2>&1 | tail -30

/content/IMM-TSF
Test - Best epoch, Loss, MSE, MAE: 37, 0.62184, 0.62184, 0.58766
Time spent: 1.96s
100%|██████████| 6/6 [00:00<00:00, 44.82it/s]
- Epoch 045, ExpID 65288
Train - Loss (one batch): 0.69318
Val - Loss, MSE, MAE: 0.79667, 0.79667, 0.61381
Test - Best epoch, Loss, MSE, MAE: 37, 0.62184, 0.62184, 0.58766
Time spent: 1.97s
100%|██████████| 6/6 [00:00<00:00, 43.85it/s]
- Epoch 046, ExpID 65288
Train - Loss (one batch): 0.70284
Val - Loss, MSE, MAE: 0.78738, 0.78738, 0.60481
Test - Best epoch, Loss, MSE, MAE: 37, 0.62184, 0.62184, 0.58766
Time spent: 1.97s
100%|██████████| 6/6 [00:00<00:00, 44.35it/s]
- Epoch 047, ExpID 65288
Train - Loss (one batch): 0.42678
Val - Loss, MSE, MAE: 0.79431, 0.79431, 0.60257
Test - Best epoch, Loss, MSE, MAE: 37, 0.62184, 0.62184, 0.58766
Time spent: 2.20s
Exp has been early stopped!
loss: 0.621839702129364
mse: 0.621839702129364
mae: 0.5876636505126953
rmse: 0.7885681390762329
mape: -0.8421215415000916
### Done ###


In [42]:
%cd /content/IMM-TSF
!python main.py \
  --dataset EPA-Air \
  --data_root /content/IMM-TSF/data \
  --model TimeMixer \
  --history 7 \
  --pred_window 7 \
  --stride 7 \
  --time_unit days \
  --split_method sample \
  --enable_text \
  --use_text_embeddings \
  --TTF_module TTF_RecAvg \
  --MMF_module MMF_GR_Add \
  --llm_model_fusion GPT2 \
  --seed 42 \
  --gpu 0 \
  --batch_size 8 \
  --lr 1e-3 \
  --epoch 50 \
  --patience 10 \
  2>&1 | tail -30

/content/IMM-TSF
Test - Best epoch, Loss, MSE, MAE: 46, 0.57839, 0.57839, 0.56117
Time spent: 2.46s
100%|██████████| 5/5 [00:00<00:00, 38.99it/s]
- Epoch 047, ExpID 60010
Train - Loss (one batch): 0.38590
Val - Loss, MSE, MAE: 0.79149, 0.79149, 0.61451
Test - Best epoch, Loss, MSE, MAE: 46, 0.57839, 0.57839, 0.56117
Time spent: 2.35s
100%|██████████| 5/5 [00:00<00:00, 39.37it/s]
- Epoch 048, ExpID 60010
Train - Loss (one batch): 0.75706
Val - Loss, MSE, MAE: 0.79650, 0.79650, 0.61598
Test - Best epoch, Loss, MSE, MAE: 46, 0.57839, 0.57839, 0.56117
Time spent: 2.31s
100%|██████████| 6/6 [00:00<00:00, 39.58it/s]
- Epoch 049, ExpID 60010
Train - Loss (one batch): 0.68529
Val - Loss, MSE, MAE: 0.78334, 0.78334, 0.61085
Test - Best epoch, Loss, MSE, MAE: 49, 0.57633, 0.57633, 0.55820
Time spent: 2.47s
loss: 0.5763250589370728
mse: 0.5763250589370728
mae: 0.5581960678100586
rmse: 0.7591607570648193
mape: -0.7283191084861755
### Done ###


# 6. TTM

In [43]:
%cd /content/IMM-TSF
!python main.py \
  --dataset EPA-Air \
  --data_root /content/IMM-TSF/data \
  --model TTM \
  --history 7 \
  --pred_window 7 \
  --stride 7 \
  --time_unit days \
  --split_method sample \
  --seed 42 \
  --gpu 0 \
  --batch_size 8 \
  --lr 1e-3 \
  --epoch 50 \
  --patience 10 \
  2>&1 | tail -40

/content/IMM-TSF
- Epoch 006, ExpID 20515
Train - Loss (one batch): 0.66434
Val - Loss, MSE, MAE: 0.87404, 0.87404, 0.63763
Test - Best epoch, Loss, MSE, MAE: 0, 0.58203, 0.58203, 0.56916
Time spent: 3.17s
100%|██████████| 6/6 [00:00<00:00, 33.68it/s]
- Epoch 007, ExpID 20515
Train - Loss (one batch): 0.47178
Val - Loss, MSE, MAE: 0.85244, 0.85244, 0.62746
Test - Best epoch, Loss, MSE, MAE: 0, 0.58203, 0.58203, 0.56916
Time spent: 3.14s
100%|██████████| 6/6 [00:00<00:00, 33.77it/s]
- Epoch 008, ExpID 20515
Train - Loss (one batch): 0.37565
Val - Loss, MSE, MAE: 0.87375, 0.87375, 0.63902
Test - Best epoch, Loss, MSE, MAE: 0, 0.58203, 0.58203, 0.56916
Time spent: 3.13s
100%|██████████| 6/6 [00:00<00:00, 33.67it/s]
- Epoch 009, ExpID 20515
Train - Loss (one batch): 0.50704
Val - Loss, MSE, MAE: 0.87745, 0.87745, 0.64019
Test - Best epoch, Loss, MSE, MAE: 0, 0.58203, 0.58203, 0.56916
Time spent: 3.17s
100%|██████████| 6/6 [00:00<00:00, 34.15it/s]
- Epoch 010, ExpID 20515
Train - Loss (one 

In [44]:
%cd /content/IMM-TSF
!python main.py \
  --dataset EPA-Air \
  --data_root /content/IMM-TSF/data \
  --model TTM \
  --history 7 \
  --pred_window 7 \
  --stride 7 \
  --time_unit days \
  --split_method sample \
  --enable_text \
  --use_text_embeddings \
  --TTF_module TTF_RecAvg \
  --MMF_module MMF_GR_Add \
  --llm_model_fusion GPT2 \
  --seed 42 \
  --gpu 0 \
  --batch_size 8 \
  --lr 1e-3 \
  --epoch 50 \
  --patience 10 \
  2>&1 | tail -40

/content/IMM-TSF
- Epoch 011, ExpID 77088
Train - Loss (one batch): 0.12426
Val - Loss, MSE, MAE: 0.88327, 0.88327, 0.66342
Test - Best epoch, Loss, MSE, MAE: 5, 0.59627, 0.59627, 0.57113
Time spent: 3.51s
100%|██████████| 5/5 [00:00<00:00, 31.34it/s]
- Epoch 012, ExpID 77088
Train - Loss (one batch): 0.34457
Val - Loss, MSE, MAE: 0.87345, 0.87345, 0.65692
Test - Best epoch, Loss, MSE, MAE: 5, 0.59627, 0.59627, 0.57113
Time spent: 3.54s
100%|██████████| 5/5 [00:00<00:00, 31.71it/s]
- Epoch 013, ExpID 77088
Train - Loss (one batch): 0.33107
Val - Loss, MSE, MAE: 0.86851, 0.86851, 0.65205
Test - Best epoch, Loss, MSE, MAE: 5, 0.59627, 0.59627, 0.57113
Time spent: 3.55s
100%|██████████| 5/5 [00:00<00:00, 31.50it/s]
- Epoch 014, ExpID 77088
Train - Loss (one batch): 0.16700
Val - Loss, MSE, MAE: 0.87853, 0.87853, 0.66202
Test - Best epoch, Loss, MSE, MAE: 5, 0.59627, 0.59627, 0.57113
Time spent: 3.53s
100%|██████████| 5/5 [00:00<00:00, 31.69it/s]
- Epoch 015, ExpID 77088
Train - Loss (one 

In [48]:
log_run('TTM', 'uni', None, None, None, 0.5820, 0.5692, 0, 'reproduction - better than paper')
log_run('TTM', 'multi', 'RecAvg', 'GR_Add', 'GPT2', 0.5963, 0.5711, 5, 'reproduction - text hurts, matches paper')

Logged: TTM (uni) MSE=0.5820 MAE=0.5692
Logged: TTM (multi) MSE=0.5963 MAE=0.5711


---
# Phase 2 — Remaining 5 Models from TIME-IMM Table 11

The 6 models already implemented above (DLinear, Informer, PatchTST, TimesNet, TimeMixer, TTM) cover only part of the paper's Table 11. The complete 11-model list is verified directly from the official `main_all.py` in the IMM-TSF repository:

```python
model_name_list = [
    "Informer", "DLinear", "PatchTST", "TimesNet", "TimeMixer",
    "TimeLLM", "TTM", "CRU", "LatentODE", "NeuralFlow", "tPatchGNN",
]
```

The 5 missing models are: **TimeLLM, CRU, LatentODE, NeuralFlow, tPatchGNN**.

Note that CRU, LatentODE, and NeuralFlow are ODE/flow-based models specifically designed for irregularly-sampled time series — they exist in the benchmark precisely because TIME-IMM tests irregularity, which is not something the standard Transformer-based models handle natively.

## Hyperparameter alignment with the existing 6 cells

Every flag below is **identical** to the existing 6 model runs (same `--history 7 --pred_window 7 --stride 7 --time_unit days --split_method sample --seed 42 --batch_size 8 --lr 1e-3 --epoch 50 --patience 10`). For multi runs we use the same `TTF_RecAvg + MMF_GR_Add + GPT2` combination your team already used for the 6 existing models. This is the only way to make the new numbers comparable to the existing 6.

## Two caveats vs the paper's exact protocol

I noticed two differences between your team's existing setup and the official `main_all.py`. **Both apply to the 6 models already done as well as the 5 new ones**, so they don't make the new runs less comparable to the existing 6 — but they do explain any gap between your numbers and the paper's headline numbers in Table 11:

1. **`--llm_model_fusion`**: Your team uses `GPT2`. The paper's `main_all.py` default is `DeepSeek` (with GPT2/BERT/Llama as commented-out alternatives). Switching to DeepSeek requires significantly more GPU memory (the README warns about needing 24GB+) — GPT2 is a reasonable practical choice for a class project.

2. **TTF/MMF combinations**: The paper iterates over both `TTF_RecAvg` and `TTF_T2V_XAttn`, and over both `MMF_GR_Add` and `MMF_XAttn_Add`. Table 11's "Multi" columns are the **best across all 4 (TTF × MMF) combinations**. Your team only runs `TTF_RecAvg + MMF_GR_Add`. So expect your "multi" numbers to be at-or-slightly-worse than the paper's "Multi" columns.

If you want to address either deviation later, it's a one-line change in the multi cells below.

## Step 1 — Sanity check the model files exist

Run this once before launching any expensive training. It just lists `models/*.py` so you can confirm `TimeLLM.py`, `CRU.py`, `LatentODE.py`, `NeuralFlow.py`, and `tPatchGNN.py` are all present in the cloned repo. If any are missing, the corresponding run cell will fail fast and you'll know to update the model name.

In [45]:
# ── Verify all 11 models exist as files in the repo ─────────────────────────
import os
models_dir = '/content/IMM-TSF/models'
print(f'=== Files in {models_dir}/ ===')
files = sorted(os.listdir(models_dir))
for f in files:
    print(f'  {f}')

print()
print('=== Checking the 11 expected model files ===')
expected = [
    'Informer.py', 'DLinear.py', 'PatchTST.py', 'TimesNet.py', 'TimeMixer.py',
    'TimeLLM.py', 'TTM.py', 'CRU.py', 'LatentODE.py', 'NeuralFlow.py', 'tPatchGNN.py',
]
missing = []
for name in expected:
    present = name in files
    mark = '✓' if present else '✗ MISSING'
    print(f'  {mark}  {name}')
    if not present:
        missing.append(name)

if missing:
    print(f'\n⚠ {len(missing)} expected model file(s) missing — check the repo version.')
    print('  Some libraries name files differently (e.g. Tpatchgnn.py vs tPatchGNN.py).')
else:
    print('\n✓ All 11 model files present. Safe to proceed.')

=== Files in /content/IMM-TSF/models/ ===
  CRU.py
  DLinear.py
  Informer.py
  LatentODE.py
  NeuralFlow.py
  PatchTST.py
  TTM.py
  TimeLLM.py
  TimeMixer.py
  TimesNet.py
  __init__.py
  __pycache__
  _old_models
  tPatchGNN.py

=== Checking the 11 expected model files ===
  ✓  Informer.py
  ✓  DLinear.py
  ✓  PatchTST.py
  ✓  TimesNet.py
  ✓  TimeMixer.py
  ✓  TimeLLM.py
  ✓  TTM.py
  ✓  CRU.py
  ✓  LatentODE.py
  ✓  NeuralFlow.py
  ✓  tPatchGNN.py

✓ All 11 model files present. Safe to proceed.


## Step 2 — Run each missing model

For each model, run the unimodal cell first. Look at the tail output for the lines:

```
loss: 0.XXXXX
mse:  0.XXXXX
mae:  0.XXXXX
```

and the `[Early stop] epoch N` line. Then run the multi cell, then paste both into the `log_run` cell and execute it.

### Model 7: TimeLLM (Jin et al. 2024)

TimeLLM reprograms a frozen LLM (the same `--llm_model_fusion` model — GPT2 in our setup) as a time series forecaster via patch reprogramming and a learnable prompt prefix. **This is the only model in the grid that uses the LLM as a backbone, not just for text fusion.** It is typically the slowest unimodal run in the table — expect each epoch to take noticeably longer than the other 10.

In [46]:
!cd /content/IMM-TSF && git checkout -- models/TimeLLM.py

%cd /content/IMM-TSF

import os, re
path = '/content/IMM-TSF/models/TimeLLM.py'
with open(path, 'r') as f:
    src = f.read()

# 🔧 File-based patch (affects the subprocess run by !python)

# 1. Fix the output projection dimension
src = re.sub(r'self\.output_projection\s*=\s*FlattenHead\([^,]+,',
             'self.output_projection = FlattenHead(self.d_llm * self.patch_nums,', src)

# 2. Fix the view/reshape shape mismatches
src = src.replace("rep_out = rep_out.view(B, N, self.patch_nums, self.d_llm)",
                  "rep_out = rep_out.reshape(B, N, -1, self.d_llm)")
src = src.replace("dec = dec.view(B, self.patch_nums, n_vars, self.d_ff)",
                  "dec = dec.reshape(B, -1, n_vars, dec.shape[-1])")
src = src.replace("dec = dec.permute(0, 2, 3, 1).reshape(B * n_vars, self.d_ff, self.patch_nums)",
                  "dec = dec.permute(0, 2, 3, 1).reshape(B * n_vars, dec.shape[-2], -1)")

with open(path, 'w') as f:
    f.write(src)

# 🚀 RUN
!python main.py \
  --dataset EPA-Air \
  --data_root /content/IMM-TSF/data \
  --model TimeLLM \
  --history 7 \
  --pred_window 7 \
  --stride 7 \
  --time_unit days \
  --split_method sample \
  --seed 42 \
  --gpu 0 \
  --batch_size 8 \
  --lr 1e-3 \
  --epoch 50 \
  --patience 10 \
  2>&1 | tail -60

/content/IMM-TSF
Train - Loss (one batch): 0.40381
Val - Loss, MSE, MAE: 0.79327, 0.79327, 0.60740
Test - Best epoch, Loss, MSE, MAE: 16, 0.60975, 0.60975, 0.57701
Time spent: 2.53s
100%|██████████| 6/6 [00:00<00:00, 27.64it/s]
- Epoch 020, ExpID 51210
Train - Loss (one batch): 0.66112
Val - Loss, MSE, MAE: 0.79370, 0.79370, 0.60931
Test - Best epoch, Loss, MSE, MAE: 16, 0.60975, 0.60975, 0.57701
Time spent: 2.53s
100%|██████████| 6/6 [00:00<00:00, 27.10it/s]
- Epoch 021, ExpID 51210
Train - Loss (one batch): 0.41755
Val - Loss, MSE, MAE: 0.79168, 0.79168, 0.61039
Test - Best epoch, Loss, MSE, MAE: 16, 0.60975, 0.60975, 0.57701
Time spent: 2.58s
100%|██████████| 6/6 [00:00<00:00, 27.16it/s]
- Epoch 022, ExpID 51210
Train - Loss (one batch): 0.47991
Val - Loss, MSE, MAE: 0.80040, 0.80040, 0.60929
Test - Best epoch, Loss, MSE, MAE: 16, 0.60975, 0.60975, 0.57701
Time spent: 2.54s
100%|██████████| 6/6 [00:00<00:00, 26.89it/s]
- Epoch 023, ExpID 51210
Train - Loss (one batch): 0.98503
Val -

In [47]:
# Fill in the values from the cell outputs above.
# Look for the lines `loss: ...`, `mse: ...`, `mae: ...` and the early-stop epoch.
log_run('TimeLLM', 'uni',   None,     None,     None,   0.6098, 0.5770, 16, 'reproduction')
# Uncomment and fill in the line below once you have the multimodal results
# log_run('TimeLLM', 'multi', 'RecAvg', 'GR_Add', 'GPT2', XXXX, XXXX, 0, 'reproduction')

Logged: TimeLLM (uni) MSE=0.6098 MAE=0.5770


# Upto here 7 models done

### Model 8: CRU — Continuous Recurrent Unit (Schirmer et al. 2022)

A continuous-time recurrent state-space model that handles irregular sampling natively via a Kalman-filter-style update. Designed for medical time series — fits the EPA-Air sparsity profile well in principle.

In [61]:
%cd /content/IMM-TSF
!python main.py \
  --dataset EPA-Air \
  --data_root /content/IMM-TSF/data \
  --model CRU \
  --history 7 \
  --pred_window 7 \
  --stride 7 \
  --time_unit days \
  --split_method sample \
  --seed 42 \
  --gpu 0 \
  --batch_size 8 \
  --lr 1e-3 \
  --epoch 50 \
  --patience 10 \
  2>&1 | tail -40

/content/IMM-TSF


In [ ]:
%cd /content/IMM-TSF
!python main.py \
  --dataset EPA-Air \
  --data_root /content/IMM-TSF/data \
  --model CRU \
  --history 7 \
  --pred_window 7 \
  --stride 7 \
  --time_unit days \
  --split_method sample \
  --enable_text \
  --use_text_embeddings \
  --TTF_module TTF_RecAvg \
  --MMF_module MMF_GR_Add \
  --llm_model_fusion GPT2 \
  --seed 42 \
  --gpu 0 \
  --batch_size 8 \
  --lr 1e-3 \
  --epoch 50 \
  --patience 10 \
  2>&1 | tail -40

In [ ]:
# Fill in the values from the cell outputs above.
# Look for the lines `loss: ...`, `mse: ...`, `mae: ...` and the early-stop epoch.
log_run('CRU', 'uni',   None,     None,     None,   XXXX, XXXX, 0, 'reproduction')
log_run('CRU', 'multi', 'RecAvg', 'GR_Add', 'GPT2', XXXX, XXXX, 0, 'reproduction')

### Model 9: LatentODE (Rubanova et al. 2019)

Models the time series latent state as the solution of a learned ODE. The encoder is an ODE-RNN that processes irregular observations directly without imputation. The original paper from NeurIPS 2019 — one of the foundational irregular-time-series methods.

In [ ]:
%cd /content/IMM-TSF
!python main.py \
  --dataset EPA-Air \
  --data_root /content/IMM-TSF/data \
  --model LatentODE \
  --history 7 \
  --pred_window 7 \
  --stride 7 \
  --time_unit days \
  --split_method sample \
  --seed 42 \
  --gpu 0 \
  --batch_size 8 \
  --lr 1e-3 \
  --epoch 50 \
  --patience 10 \
  2>&1 | tail -40

In [ ]:
%cd /content/IMM-TSF
!python main.py \
  --dataset EPA-Air \
  --data_root /content/IMM-TSF/data \
  --model LatentODE \
  --history 7 \
  --pred_window 7 \
  --stride 7 \
  --time_unit days \
  --split_method sample \
  --enable_text \
  --use_text_embeddings \
  --TTF_module TTF_RecAvg \
  --MMF_module MMF_GR_Add \
  --llm_model_fusion GPT2 \
  --seed 42 \
  --gpu 0 \
  --batch_size 8 \
  --lr 1e-3 \
  --epoch 50 \
  --patience 10 \
  2>&1 | tail -40

In [ ]:
# Fill in the values from the cell outputs above.
# Look for the lines `loss: ...`, `mse: ...`, `mae: ...` and the early-stop epoch.
log_run('LatentODE', 'uni',   None,     None,     None,   XXXX, XXXX, 0, 'reproduction')
log_run('LatentODE', 'multi', 'RecAvg', 'GR_Add', 'GPT2', XXXX, XXXX, 0, 'reproduction')

### Model 10: NeuralFlow (Biloš et al. 2021)

Replaces the ODE solver in LatentODE with a learned bijective flow that gives a closed-form solution for the latent dynamics. This is what the `stribor` package in `requirements.txt` is for — it provides the flow primitives. Substantially faster than LatentODE in practice.

In [ ]:
%cd /content/IMM-TSF
!python main.py \
  --dataset EPA-Air \
  --data_root /content/IMM-TSF/data \
  --model NeuralFlow \
  --history 7 \
  --pred_window 7 \
  --stride 7 \
  --time_unit days \
  --split_method sample \
  --seed 42 \
  --gpu 0 \
  --batch_size 8 \
  --lr 1e-3 \
  --epoch 50 \
  --patience 10 \
  2>&1 | tail -40

In [ ]:
%cd /content/IMM-TSF
!python main.py \
  --dataset EPA-Air \
  --data_root /content/IMM-TSF/data \
  --model NeuralFlow \
  --history 7 \
  --pred_window 7 \
  --stride 7 \
  --time_unit days \
  --split_method sample \
  --enable_text \
  --use_text_embeddings \
  --TTF_module TTF_RecAvg \
  --MMF_module MMF_GR_Add \
  --llm_model_fusion GPT2 \
  --seed 42 \
  --gpu 0 \
  --batch_size 8 \
  --lr 1e-3 \
  --epoch 50 \
  --patience 10 \
  2>&1 | tail -40

In [ ]:
# Fill in the values from the cell outputs above.
# Look for the lines `loss: ...`, `mse: ...`, `mae: ...` and the early-stop epoch.
log_run('NeuralFlow', 'uni',   None,     None,     None,   XXXX, XXXX, 0, 'reproduction')
log_run('NeuralFlow', 'multi', 'RecAvg', 'GR_Add', 'GPT2', XXXX, XXXX, 0, 'reproduction')

### Model 11: tPatchGNN (Zhang et al. 2024)

Temporal patching combined with a learnable inter-sensor graph neural network. **Confirmed present in the repo** — Cell 8 above showed the import line `from models.tPatchGNN import tPatchGNN` in the original failing traceback before `reformer_pytorch` was installed. This is the most recent of the 11 models and one of the strongest baselines on irregular multivariate data.

In [ ]:
%cd /content/IMM-TSF
!python main.py \
  --dataset EPA-Air \
  --data_root /content/IMM-TSF/data \
  --model tPatchGNN \
  --history 7 \
  --pred_window 7 \
  --stride 7 \
  --time_unit days \
  --split_method sample \
  --seed 42 \
  --gpu 0 \
  --batch_size 8 \
  --lr 1e-3 \
  --epoch 50 \
  --patience 10 \
  2>&1 | tail -40

In [ ]:
%cd /content/IMM-TSF
!python main.py \
  --dataset EPA-Air \
  --data_root /content/IMM-TSF/data \
  --model tPatchGNN \
  --history 7 \
  --pred_window 7 \
  --stride 7 \
  --time_unit days \
  --split_method sample \
  --enable_text \
  --use_text_embeddings \
  --TTF_module TTF_RecAvg \
  --MMF_module MMF_GR_Add \
  --llm_model_fusion GPT2 \
  --seed 42 \
  --gpu 0 \
  --batch_size 8 \
  --lr 1e-3 \
  --epoch 50 \
  --patience 10 \
  2>&1 | tail -40

In [ ]:
# Fill in the values from the cell outputs above.
# Look for the lines `loss: ...`, `mse: ...`, `mae: ...` and the early-stop epoch.
log_run('tPatchGNN', 'uni',   None,     None,     None,   XXXX, XXXX, 0, 'reproduction')
log_run('tPatchGNN', 'multi', 'RecAvg', 'GR_Add', 'GPT2', XXXX, XXXX, 0, 'reproduction')

## Step 3 — Full 11-model comparison vs paper Table 11

Read all logged runs from `runs.jsonl` and print a side-by-side table against the paper's published numbers. Models still pending will show `(not yet run)`.

In [ ]:
# ── Aggregate all 11 models from runs.jsonl, side-by-side with paper Table 11 ──
import json
from pathlib import Path

RESULTS_FILE = '/content/drive/MyDrive/STAT8240_EPA_Project/results/runs.jsonl'

# Read every entry, keeping the LAST run per (model, modal) so re-runs overwrite
runs = {}
if Path(RESULTS_FILE).exists():
    with open(RESULTS_FILE) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            r = json.loads(line)
            key = (r['model'], r['modal'])
            runs[key] = r
else:
    print(f'WARNING: {RESULTS_FILE} not found yet — log some runs first.')

# Paper Table 11 reference values (verified from screenshot)
PAPER = {
    'Informer':   {'uni': (0.6301, 0.5983), 'multi': (0.5812, 0.5728)},
    'DLinear':    {'uni': (0.5361, 0.5279), 'multi': (0.5223, 0.5223)},
    'PatchTST':   {'uni': (0.6196, 0.5947), 'multi': (0.6204, 0.5992)},
    'TimesNet':   {'uni': (0.5599, 0.5524), 'multi': (0.5892, 0.5650)},
    'TimeMixer':  {'uni': (0.6086, 0.5770), 'multi': (0.5641, 0.5663)},
    'TimeLLM':    {'uni': (0.5835, 0.5643), 'multi': (0.5334, 0.5303)},
    'TTM':        {'uni': (0.6002, 0.5653), 'multi': (0.6218, 0.5761)},
    'CRU':        {'uni': (0.7026, 0.6306), 'multi': (0.7982, 0.6739)},
    'LatentODE':  {'uni': (0.8025, 0.6665), 'multi': (0.7556, 0.6523)},
    'NeuralFlow': {'uni': (0.7821, 0.6488), 'multi': (0.8202, 0.6790)},
    'tPatchGNN':  {'uni': (0.6258, 0.6022), 'multi': (0.5840, 0.5793)},
}

# Print order — same as paper Table 11
ALL_MODELS = list(PAPER.keys())

print('=' * 102)
print(f'EPA-Air | TIME-IMM Table 11 reproduction (seed=42, GPT2 fusion, RecAvg+GR_Add)')
print('=' * 102)
print(f'{"Model":<13} {"Modal":<6}  {"Ours MSE":>9} {"Paper MSE":>10} {"Δ":>8}    '
      f'{"Ours MAE":>9} {"Paper MAE":>10} {"Δ":>8}')
print('-' * 102)
for m in ALL_MODELS:
    for modal in ('uni', 'multi'):
        r  = runs.get((m, modal))
        pp = PAPER[m][modal]
        if r is None:
            ours_mse = '   —    '
            ours_mae = '   —    '
            d_mse = '   —  '
            d_mae = '   —  '
        else:
            ours_mse = f'{r["mse"]:.4f}'
            ours_mae = f'{r["mae"]:.4f}'
            d_mse = f'{r["mse"]-pp[0]:+.4f}'
            d_mae = f'{r["mae"]-pp[1]:+.4f}'
        label = m if modal == 'uni' else ''
        print(f'{label:<13} {modal:<6}  {ours_mse:>9} {pp[0]:>10.4f} {d_mse:>8}    '
              f'{ours_mae:>9} {pp[1]:>10.4f} {d_mae:>8}')
    print('-' * 102)

print()
done = sum(1 for m in ALL_MODELS for modal in ('uni','multi') if (m,modal) in runs)
print(f'Logged runs: {done}/22  ({len(ALL_MODELS)} models × 2 modes)')